# Fig 1B — Allele-flip QC benchmark (data processing)

Allele-flip simulation: power / FDR of kriging-based allele QC against the other QC methods, across flip rate, number of causal variants and NCP. Bootstrap 95% CIs over replicates.

**Output:** `figure_data/Fig1B.Allele_Flip_Kriging.rds` (+ `.csv`)

In [1]:
source("../../R/utils.R")

data_name <- "Fig1B.Allele_Flip_Kriging"
n_gwas    <- 800                      # GTEx eQTL sample size; NCP = n * h2g

In [2]:
## ---- load + assemble (DENTIST/DEN_QC removed; DENTIST + RAISS kept) --------
result <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip/summary/result_susie_bias_summary_summary.rds") %>%
  filter(method != "kriging_outlier")
result_kriging <- result %>% filter(method != "original_res") %>%
  mutate(qc = method) %>% mutate(method = "gtex_flip")

carma         <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_CARMA_Flip/summary/result_single_QC_summary.rds")             %>% mutate(qc = "CARMA")
dentist_raiss <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_DEN_Flip//summary/result_DEN_SLA_summary_DENTIST_RAISS.rds") %>% mutate(qc = "DEN_RAISS")
slalom_raiss  <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_DEN_Flip//summary/result_DEN_SLA_summary_SLALOM_RAISS.rds")  %>% mutate(qc = "SLALOM_RAISS")
rsparse_pro   <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_RSP_Flip//summary/result_single_QC_summary.rds")              %>% mutate(qc = "Rsparse_Pro")

## per-replicate rows (each row = one replicate within a cell)
cell_reps <- bind_rows(carma, dentist_raiss, slalom_raiss, rsparse_pro, result_kriging) %>%
  filter(method == "gtex_flip") %>%
  mutate(h2g       = str_extract(input_file, "h2g_[0-9.]+"),
         miss_prop = str_extract(input_file, "miss_prop_[0-9.]+"),
         causal_k  = total_var_number)          # <- survives group_modify (not a grouping key)

In [3]:
## ---- bootstrap over replicates within each cell (boot_cell() in R/utils.R) --
## point estimate = pooled ratio; CI = 2.5/97.5% percentiles of the pooled
## metric across B resamples of rows.
B <- 1000
set.seed(1)

summary_ci <- cell_reps %>%
  group_by(qc, method, h2g, miss_prop, total_var_number) %>%
  group_modify(~ boot_cell(.x, B = B)) %>%
  ungroup()

In [4]:
## ---- plot-ready labels / facets ---------------------------------------------
plot_df <- summary_ci %>%
  mutate(
    miss_prop_num = as.numeric(str_remove(miss_prop, "miss_prop_")),
    miss_prop = fct_reorder(paste0("Flip Rate ", formatC(miss_prop_num * 100, format = "g"), "%"),
                            miss_prop_num),
    h2g_num = as.numeric(str_remove(h2g, "h2g_")),
    ncp_num = n_gwas * h2g_num,                       # single-variant chi-sq NCP
    ncp = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 1)), ncp_num),
    qc = recode(qc,
      flipped_res           = "SuSiE-RSS",
      CARMA                 = "CARMA",
      DEN_RAISS             = "DENTIST + RAISS",
      SLALOM_RAISS          = "SLALOM + RAISS",
      Rsparse_Pro           = "RSparsePro",
      kriging_res           = "Kriging",         # default method
      .default = as.character(qc)),
    qc = factor(qc, levels = names(pal_kriging))
  ) %>% filter(miss_prop_num != 0.1)
stopifnot(!any(is.na(plot_df$qc)))

save_data(plot_df, data_name)